# Imports and Data Load

In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
import spacy
import re
import nltk
import numpy as np
nltk.download("stopwords")
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from textblob import TextBlob
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics import classification_report, accuracy_score
from sklearn.decomposition import TruncatedSVD
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, SimpleRNN, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
!pip install gensim
import gensim.downloader as api


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [20]:
train_data = pd.read_csv('https://drive.google.com/uc?id=1-fZUIY8HJ_P6M8bQ2LUITwR-G_zgIDK5')
test_data = pd.read_csv('https://drive.google.com/uc?id=1k-mFYjbegv0OW5NgF1f0gMX8bAQgSL6V')

In [21]:
X_train_full = train_data.drop(columns=['Sentiment'])
y_train_full = train_data['Sentiment']
X_test = test_data

X_train, X_val, y_train, y_val = train_test_split(
    train_data['Text'],
    train_data['Sentiment'],
    test_size=0.2,
    random_state=42,
    stratify=train_data['Sentiment']
)

# Preprocess

In [22]:
class Preprocess(BaseEstimator, TransformerMixin):
    def __init__(self,
                 lowercase=True,
                 lemmatize=True,
                 pos_filter=False,
                 entity_masking=False):
        self.lowercase = lowercase
        self.lemmatize = lemmatize
        self.pos_filter = pos_filter
        self.entity_masking = entity_masking
        self.stop_words = set(stopwords.words("english"))
        self.lemmatizer = WordNetLemmatizer()
        self.spacy_nlp = spacy.load("en_core_web_sm")

    def mask_entities(self, text):
        doc = self.spacy_nlp(text)
        for ent in doc.ents:
            text = text.replace(ent.text, f"<{ent.label_}>")
        return text

    def preprocess_text(self, text):
        # Lowercase
        if self.lowercase:
            text = text.lower()

        # Mask entities
        if self.entity_masking:
            text = self.mask_entities(text)

        # Remove unwanted characters
        text = re.sub(r"[^a-z\s]", " ", text)

        tokens = text.split()

        # POS Tagger filtering
        if self.pos_filter:
            doc = self.spacy_nlp(" ".join(tokens))
            tokens = [token.text for token in doc if token.pos_ in ["ADJ", "ADV", "VERB", "NOUN"]]

        # Stopword elimination
        tokens = [w for w in tokens if w not in self.stop_words]

        # Lemmatization
        if self.lemmatize:
            tokens = [self.lemmatizer.lemmatize(w) for w in tokens]

        return " ".join(tokens)

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        return [self.preprocess_text(text) for text in X]

# Load Pretrained Word2Vec, Glove, and Fasttext

### Word2Vec

In [33]:
w2v_model = api.load("word2vec-google-news-300")

embedding_dim_w2v = 300

max_len = 100
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq   = tokenizer.texts_to_sequences(X_val)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_val_pad   = pad_sequences(X_val_seq, maxlen=max_len)

label_enc = LabelEncoder()
y_train_enc = label_enc.fit_transform(y_train)
y_val_enc   = label_enc.transform(y_val)
num_classes = len(label_enc.classes_)

word_index = tokenizer.word_index
embedding_matrix_w2v = np.zeros((len(word_index)+1, embedding_dim_w2v))
for word, i in word_index.items():
    if word in w2v_model:
        embedding_matrix_w2v[i] = w2v_model[word]

### Glove

In [34]:
glove_model = api.load("glove-wiki-gigaword-100")

embedding_dim_glove = 100

max_len = 100
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq   = tokenizer.texts_to_sequences(X_val)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_val_pad   = pad_sequences(X_val_seq, maxlen=max_len)

label_enc = LabelEncoder()
y_train_enc = label_enc.fit_transform(y_train)
y_val_enc   = label_enc.transform(y_val)
num_classes = len(label_enc.classes_)

word_index = tokenizer.word_index
embedding_matrix_glove = np.zeros((len(word_index)+1, embedding_dim_glove))
for word, i in word_index.items():
    if word in glove_model:
        embedding_matrix_glove[i] = glove_model[word]

### Fasttext

In [35]:
fasttext_model = api.load("fasttext-wiki-news-subwords-300")

embedding_dim_fasttext = 300

max_len = 100
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq   = tokenizer.texts_to_sequences(X_val)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_val_pad   = pad_sequences(X_val_seq, maxlen=max_len)

label_enc = LabelEncoder()
y_train_enc = label_enc.fit_transform(y_train)
y_val_enc   = label_enc.transform(y_val)
num_classes = len(label_enc.classes_)

word_index = tokenizer.word_index
embedding_matrix_fasttext = np.zeros((len(word_index)+1, embedding_dim_fasttext))
for word, i in word_index.items():
    if word in fasttext_model:
        embedding_matrix_fasttext[i] = fasttext_model[word]

# Prepare BoW, TF-IDF, and SVD

In [36]:
preprocessor = Preprocess(
    lowercase=True,
    lemmatize=True,
    pos_filter=True,
    entity_masking=True
)

X_train_clean = preprocessor.transform(X_train)
X_val_clean   = preprocessor.transform(X_val)

features = FeatureUnion([
    ("bow", CountVectorizer(max_features=1000, ngram_range=(1,1))),
    ("tfidf", TfidfVectorizer(max_features=2000, ngram_range=(1,2), sublinear_tf=True)),
])

X_train_features = features.fit_transform(X_train_clean)
X_val_features   = features.transform(X_val_clean)

svd_dim = 300
svd = TruncatedSVD(n_components=svd_dim, random_state=42)
X_train_svd = svd.fit_transform(X_train_features)
X_val_svd   = svd.transform(X_val_features)

X_train_seq = np.expand_dims(X_train_svd, axis=-1)
X_val_seq   = np.expand_dims(X_val_svd, axis=-1)

label_enc = LabelEncoder()
y_train_enc = label_enc.fit_transform(y_train)
y_val_enc   = label_enc.transform(y_val)

num_classes = len(label_enc.classes_)

# Feature Extraction & Machine Learning Algorithm Analysis

### CNN

Word2Vec

In [37]:
cnn_model_w2v = Sequential([
    Embedding(len(word_index)+1, embedding_dim_w2v, weights=[embedding_matrix_w2v],
              input_length=max_len, trainable=False),
    Conv1D(128, 5, activation="relu"),
    GlobalMaxPooling1D(),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dense(num_classes, activation="softmax")
])

cnn_model_w2v.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

cnn_model_w2v.fit(
    X_train_pad,
    y_train_enc,
    validation_data=(X_val_pad, y_val_enc),
    epochs=5,
    batch_size=32
  )

val_loss, val_acc = cnn_model_w2v.evaluate(X_val_pad, y_val_enc)
print("CNN with Word2Vec Validation Accuracy:", val_acc)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 5s 149ms/step - accuracy: 0.7946 - loss: 0.5007 - val_accuracy: 0.8390 - val_loss: 0.4270
Epoch 2/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8290 - loss: 0.4020 - val_accuracy: 0.8390 - val_loss: 0.4069
Epoch 3/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8455 - loss: 0.3187 - val_accuracy: 0.8305 - val_loss: 0.3719
Epoch 4/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8718 - loss: 0.2619 - val_accuracy: 0.8644 - val_loss: 0.3457
Epoch 5/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9347 - loss: 0.2000 - val_accuracy: 0.8729 - val_loss: 0.3227
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8492 - loss: 0.3613 
CNN with Word2Vec Validation Accuracy: 0.8728813529014587


Glove

In [38]:
cnn_model_glove = Sequential([
    Embedding(len(word_index)+1, embedding_dim_glove, weights=[embedding_matrix_glove],
              input_length=max_len, trainable=False),
    Conv1D(128, 5, activation="relu"),
    GlobalMaxPooling1D(),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dense(num_classes, activation="softmax")
])

cnn_model_glove.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

cnn_model_glove.fit(
    X_train_pad,
    y_train_enc,
    validation_data=(X_val_pad, y_val_enc),
    epochs=5,
    batch_size=32
  )

val_loss, val_acc = cnn_model_glove.evaluate(X_val_pad, y_val_enc)
print("CNN with Glove Validation Accuracy:", val_acc)

Epoch 1/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 5s 191ms/step - accuracy: 0.8134 - loss: 0.5027 - val_accuracy: 0.8390 - val_loss: 0.4819
Epoch 2/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8343 - loss: 0.4519 - val_accuracy: 0.8390 - val_loss: 0.4162
Epoch 3/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8525 - loss: 0.3460 - val_accuracy: 0.8475 - val_loss: 0.4094
Epoch 4/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8860 - loss: 0.3125 - val_accuracy: 0.8475 - val_loss: 0.4018
Epoch 5/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8888 - loss: 0.2325 - val_accuracy: 0.8559 - val_loss: 0.3881
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8559 - loss: 0.3998 
CNN with Glove Validation Accuracy: 0.8559321761131287


Fasttext

In [39]:
cnn_model_fasttext = Sequential([
    Embedding(len(word_index)+1, embedding_dim_fasttext, weights=[embedding_matrix_fasttext],
              input_length=max_len, trainable=False),
    Conv1D(128, 5, activation="relu"),
    GlobalMaxPooling1D(),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dense(num_classes, activation="softmax")
])

cnn_model_fasttext.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

cnn_model_fasttext.fit(
    X_train_pad,
    y_train_enc,
    validation_data=(X_val_pad, y_val_enc),
    epochs=5,
    batch_size=32
  )

val_loss, val_acc = cnn_model_fasttext.evaluate(X_val_pad, y_val_enc)
print("CNN with Fasttext Validation Accuracy:", val_acc)

Epoch 1/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 4s 157ms/step - accuracy: 0.8459 - loss: 0.4784 - val_accuracy: 0.8390 - val_loss: 0.4365
Epoch 2/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8608 - loss: 0.3832 - val_accuracy: 0.8390 - val_loss: 0.4346
Epoch 3/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8529 - loss: 0.3759 - val_accuracy: 0.8390 - val_loss: 0.4271
Epoch 4/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8324 - loss: 0.3711 - val_accuracy: 0.8390 - val_loss: 0.4195
Epoch 5/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8316 - loss: 0.3593 - val_accuracy: 0.8390 - val_loss: 0.4124
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8335 - loss: 0.4248 
CNN with Fasttext Validation Accuracy: 0.8389830589294434


SVD

In [40]:
cnn_model_svd = Sequential([
    Conv1D(128, 5, activation="relu", input_shape=(svd_dim, 1)),
    GlobalMaxPooling1D(),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dense(num_classes, activation="softmax")
])

cnn_model_svd.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

history = cnn_model_svd.fit(
    X_train_seq, y_train_enc,
    validation_data=(X_val_seq, y_val_enc),
    epochs=5,
    batch_size=32
)

val_loss, val_acc = cnn_model_svd.evaluate(X_val_cnn, y_val_enc)
print("CNN With BoW, TF-IDF, and SVD Validation Accuracy:", val_acc)

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


15/15 ━━━━━━━━━━━━━━━━━━━━ 4s 126ms/step - accuracy: 0.7812 - loss: 0.6050 - val_accuracy: 0.8390 - val_loss: 0.5017
Epoch 2/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8393 - loss: 0.4417 - val_accuracy: 0.8390 - val_loss: 0.4656
Epoch 3/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8251 - loss: 0.4867 - val_accuracy: 0.8390 - val_loss: 0.4744
Epoch 4/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8417 - loss: 0.4580 - val_accuracy: 0.8390 - val_loss: 0.4708
Epoch 5/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8266 - loss: 0.4758 - val_accuracy: 0.8390 - val_loss: 0.4735
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8335 - loss: 0.4861
CNN With BoW, TF-IDF, and SVD Validation Accuracy: 0.8389830589294434


### RNN

Word2Vec

In [41]:
rnn_model_w2v = Sequential([
    Embedding(len(word_index)+1, embedding_dim_w2v, weights=[embedding_matrix_w2v],
              input_length=max_len, trainable=False),
    SimpleRNN(128),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dense(num_classes, activation="softmax")
])

rnn_model_w2v.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

rnn_model_w2v.fit(
    X_train_pad,
    y_train_enc,
    validation_data=(X_val_pad, y_val_enc),
    epochs=5,
    batch_size=32
)

val_loss, val_acc = rnn_model_w2v.evaluate(X_val_pad, y_val_enc)
print("RNN with Word2Vec Validation Accuracy:", val_acc)

Epoch 1/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 6s 174ms/step - accuracy: 0.7610 - loss: 0.5300 - val_accuracy: 0.8390 - val_loss: 0.4578
Epoch 2/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.8336 - loss: 0.4337 - val_accuracy: 0.8051 - val_loss: 0.5141
Epoch 3/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8542 - loss: 0.3923 - val_accuracy: 0.8390 - val_loss: 0.4527
Epoch 4/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8512 - loss: 0.3334 - val_accuracy: 0.8475 - val_loss: 0.4551
Epoch 5/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8957 - loss: 0.2542 - val_accuracy: 0.8305 - val_loss: 0.4740
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8155 - loss: 0.5099
RNN with Word2Vec Validation Accuracy: 0.8305084705352783


Glove

In [42]:
rnn_model_glove = Sequential([
    Embedding(len(word_index)+1, embedding_dim_glove, weights=[embedding_matrix_glove],
              input_length=max_len, trainable=False),
    SimpleRNN(128),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dense(num_classes, activation="softmax")
])

rnn_model_glove.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

rnn_model_glove.fit(
    X_train_pad,
    y_train_enc,
    validation_data=(X_val_pad, y_val_enc),
    epochs=5,
    batch_size=32
)

val_loss, val_acc = rnn_model_glove.evaluate(X_val_pad, y_val_enc)
print("RNN with Glove Validation Accuracy:", val_acc)

Epoch 1/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 7s 292ms/step - accuracy: 0.6822 - loss: 0.6506 - val_accuracy: 0.8390 - val_loss: 0.4639
Epoch 2/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8406 - loss: 0.4291 - val_accuracy: 0.8305 - val_loss: 0.4434
Epoch 3/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8331 - loss: 0.4317 - val_accuracy: 0.8390 - val_loss: 0.4488
Epoch 4/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8571 - loss: 0.3548 - val_accuracy: 0.8475 - val_loss: 0.4345
Epoch 5/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8546 - loss: 0.3511 - val_accuracy: 0.8390 - val_loss: 0.4451
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8335 - loss: 0.4640
RNN with Glove Validation Accuracy: 0.8389830589294434


Fasttext

In [43]:
rnn_model_fasttext = Sequential([
    Embedding(len(word_index)+1, embedding_dim_fasttext, weights=[embedding_matrix_fasttext],
              input_length=max_len, trainable=False),
    SimpleRNN(128),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dense(num_classes, activation="softmax")
])

rnn_model_fasttext.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

rnn_model_fasttext.fit(
    X_train_pad,
    y_train_enc,
    validation_data=(X_val_pad, y_val_enc),
    epochs=5,
    batch_size=32
)

val_loss, val_acc = rnn_model_fasttext.evaluate(X_val_pad, y_val_enc)
print("RNN with Fasttext Validation Accuracy:", val_acc)

Epoch 1/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 5s 172ms/step - accuracy: 0.7360 - loss: 0.5333 - val_accuracy: 0.8390 - val_loss: 0.4619
Epoch 2/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.8484 - loss: 0.4502 - val_accuracy: 0.8390 - val_loss: 0.4413
Epoch 3/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8312 - loss: 0.4667 - val_accuracy: 0.8390 - val_loss: 0.4391
Epoch 4/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8315 - loss: 0.4518 - val_accuracy: 0.8390 - val_loss: 0.4398
Epoch 5/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8378 - loss: 0.4034 - val_accuracy: 0.8390 - val_loss: 0.4758
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8335 - loss: 0.4829
RNN with Fasttext Validation Accuracy: 0.8389830589294434


SVD

In [44]:
rnn_model_svd = Sequential([
    SimpleRNN(128, activation="tanh", input_shape=(svd_dim, 1)),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dense(num_classes, activation="softmax")
])

rnn_model_svd.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

rnn_model_svd.fit(
    X_train_seq,
    y_train_enc,
    validation_data=(X_val_seq, y_val_enc),
    epochs=5,
    batch_size=32
)

rnn_val_loss, rnn_val_acc = rnn_model_svd.evaluate(X_val_seq, y_val_enc)
print("RNN with BoW, TF-IDF, and SVD Validation Accuracy:", rnn_val_acc)

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


15/15 ━━━━━━━━━━━━━━━━━━━━ 6s 176ms/step - accuracy: 0.5751 - loss: 0.6324 - val_accuracy: 0.8390 - val_loss: 0.4441
Epoch 2/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.8350 - loss: 0.4949 - val_accuracy: 0.8390 - val_loss: 0.4471
Epoch 3/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.8487 - loss: 0.4330 - val_accuracy: 0.8390 - val_loss: 0.4598
Epoch 4/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.8633 - loss: 0.4228 - val_accuracy: 0.8390 - val_loss: 0.4582
Epoch 5/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.8198 - loss: 0.4862 - val_accuracy: 0.8390 - val_loss: 0.4419
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8335 - loss: 0.4503
RNN with BoW, TF-IDF, and SVD Validation Accuracy: 0.8389830589294434


### LSTM

Word2Vec

In [45]:
lstm_model_w2v = Sequential([
    Embedding(len(word_index)+1, embedding_dim_w2v, weights=[embedding_matrix_w2v],
              input_length=max_len, trainable=False),
    LSTM(128),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dense(num_classes, activation="softmax")
])

lstm_model_w2v.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

lstm_model_w2v.fit(
    X_train_pad,
    y_train_enc,
    validation_data=(X_val_pad, y_val_enc),
    epochs=5,
    batch_size=32
)

val_loss, val_acc = lstm_model_w2v.evaluate(X_val_pad, y_val_enc)
print("LSTM with Word2Vec Validation Accuracy:", val_acc)

Epoch 1/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.8148 - loss: 0.5576 - val_accuracy: 0.8390 - val_loss: 0.4235
Epoch 2/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8296 - loss: 0.4103 - val_accuracy: 0.8390 - val_loss: 0.3820
Epoch 3/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8709 - loss: 0.3167 - val_accuracy: 0.8644 - val_loss: 0.3623
Epoch 4/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8714 - loss: 0.3049 - val_accuracy: 0.8898 - val_loss: 0.3276
Epoch 5/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9246 - loss: 0.2515 - val_accuracy: 0.8898 - val_loss: 0.3056
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8674 - loss: 0.3322
LSTM with Word2Vec Validation Accuracy: 0.8898305296897888


Glove

In [46]:
lstm_model_glove = Sequential([
    Embedding(len(word_index)+1, embedding_dim_glove, weights=[embedding_matrix_glove],
              input_length=max_len, trainable=False),
    LSTM(128),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dense(num_classes, activation="softmax")
])

lstm_model_glove.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

lstm_model_glove.fit(
    X_train_pad,
    y_train_enc,
    validation_data=(X_val_pad, y_val_enc),
    epochs=5,
    batch_size=32
)

val_loss, val_acc = lstm_model_glove.evaluate(X_val_pad, y_val_enc)
print("LSTM with Glove Validation Accuracy:", val_acc)

Epoch 1/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.7390 - loss: 0.5561 - val_accuracy: 0.8390 - val_loss: 0.4382
Epoch 2/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8428 - loss: 0.4097 - val_accuracy: 0.8390 - val_loss: 0.4216
Epoch 3/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8614 - loss: 0.3622 - val_accuracy: 0.8390 - val_loss: 0.4141
Epoch 4/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8488 - loss: 0.3720 - val_accuracy: 0.8475 - val_loss: 0.4034
Epoch 5/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8745 - loss: 0.3141 - val_accuracy: 0.8390 - val_loss: 0.4083
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8148 - loss: 0.4235
LSTM with Glove Validation Accuracy: 0.8389830589294434


Fasttext

In [47]:
lstm_model_fasttext = Sequential([
    Embedding(len(word_index)+1, embedding_dim_fasttext, weights=[embedding_matrix_fasttext],
              input_length=max_len, trainable=False),
    LSTM(128),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dense(num_classes, activation="softmax")
])

lstm_model_fasttext.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

lstm_model_fasttext.fit(
    X_train_pad,
    y_train_enc,
    validation_data=(X_val_pad, y_val_enc),
    epochs=5,
    batch_size=32
)

val_loss, val_acc = lstm_model_fasttext.evaluate(X_val_pad, y_val_enc)
print("LSTM with Fasttext Validation Accuracy:", val_acc)

Epoch 1/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 0.7276 - loss: 0.5882 - val_accuracy: 0.8390 - val_loss: 0.4502
Epoch 2/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8287 - loss: 0.4507 - val_accuracy: 0.8390 - val_loss: 0.4425
Epoch 3/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8408 - loss: 0.4176 - val_accuracy: 0.8390 - val_loss: 0.4332
Epoch 4/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8522 - loss: 0.3825 - val_accuracy: 0.8390 - val_loss: 0.4240
Epoch 5/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8607 - loss: 0.3532 - val_accuracy: 0.8390 - val_loss: 0.4141
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8335 - loss: 0.4280
LSTM with Fasttext Validation Accuracy: 0.8389830589294434


SVD

In [48]:
lstm_model_svd = Sequential([
    LSTM(128, input_shape=(svd_dim, 1)),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dense(num_classes, activation="softmax")
])

lstm_model_svd.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

lstm_model_svd.fit(
    X_train_seq,
    y_train_enc,
    validation_data=(X_val_seq, y_val_enc),
    epochs=5,
    batch_size=32
  )

lstm_val_loss, lstm_val_acc = lstm_model_svd.evaluate(X_val_seq, y_val_enc)
print("LSTM with BoW, TF-IDF, and SVD Validation Accuracy:", lstm_val_acc)

Epoch 1/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - accuracy: 0.7308 - loss: 0.6518 - val_accuracy: 0.8390 - val_loss: 0.4931
Epoch 2/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8274 - loss: 0.4932 - val_accuracy: 0.8390 - val_loss: 0.4529
Epoch 3/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.8302 - loss: 0.4647 - val_accuracy: 0.8390 - val_loss: 0.4419
Epoch 4/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8400 - loss: 0.4453 - val_accuracy: 0.8390 - val_loss: 0.4415
Epoch 5/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8544 - loss: 0.4195 - val_accuracy: 0.8390 - val_loss: 0.4430
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8335 - loss: 0.4513
LSTM with BoW, TF-IDF, and SVD Validation Accuracy: 0.8389830589294434
